# Hybrid GraphRAG: Qdrant vectors + knowledge graph + community reports

This is **step 6**, hybrid retrieval and generation, using everything built so far:

| Piece | From | What it's good at |
|---|---|---|
| **Vector search** over chunks (dense + BM25 → ColBERT rerank) | `okf_chunks` in Qdrant (`01_Chunking.ipynb`) | finding the right passage even when the question's words aren't entity names |
| **Graph walk** from the question's entities, plus facts from the chunks Qdrant found | `graph_facts_v2.jsonl` (`03_knowlegde_graph.ipynb`) | precise facts about named things, and links between them |
| **Community reports** | `okf_communities` in Qdrant (`05_graph_communities.ipynb`) | the big picture: "which papers discuss X", "what are the main risks" |

`04_graph_query.ipynb` stays as the graph-only baseline, and `02_OKF_query.ipynb` as the OKF baseline. Same model, same 18,000-character context budget, same questions.

### How a question is answered

```
question ──► split two-part questions ("what are the steps of RLHF, and how did RLHF affect calibration?")
   │
   ├─► A. Qdrant chunk search (each part)                       → ranked chunk list(s)
   ├─► B. entities in the question → graph walk
   │      + facts extracted from the chunks A found                → ranked facts → their chunks
   ├─► C. fuse A and B with Reciprocal Rank Fusion (RRF)          → the passages
   ├─► D. Qdrant search over community reports
   │      + the reports of the question's entities, fused by RRF  → 2 overview reports
   └─► context = reports + facts + passages → qwen3-4b → answer citing chunk IDs and report IDs
```

## 1. Setup

The switches let you turn each part off, to see what it adds (section 12 does that for you):
- `USE_VECTORS`: Qdrant chunk search
- `USE_GRAPH`: graph walk + fact search
- `USE_COMMUNITIES`: community reports
- `SPLIT_QUESTIONS`: search each half of a two-part question separately

In [1]:
import os
from pathlib import Path
if Path.cwd().name == "notebooks":          # run from the project root, wherever Jupyter starts
    os.chdir(Path.cwd().parent)

import ast
import json
import math
import os
import re
import time
from collections import Counter
from pathlib import Path

import networkx as nx
import pandas as pd
import requests
from dotenv import load_dotenv
from qdrant_client import QdrantClient, models

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FACTS_PATH = PROJECT_ROOT / "artifacts" / "graph_facts_v2.jsonl"
ENTITIES_PATH = PROJECT_ROOT / "artifacts" / "graph_entities_v2.csv"
REPORTS_PATH = PROJECT_ROOT / "artifacts" / "graph_communities.jsonl"
CHUNKS_PATH = PROJECT_ROOT / "data" / "processed" / "chunks.jsonl"

# LLM: same as 02_OKF_query.ipynb and 04_graph_query.ipynb
OLLAMA_URL = "http://localhost:11434"
MODEL_NAME = "qwen3:4b-instruct-2507-q4_K_M"
NUM_CTX = 8192
MAX_ANSWER_TOKENS = 700
CONTEXT_CHARS = 18000

# Qdrant: same collection and models as 01_Chunking.ipynb
CHUNK_COLLECTION = "okf_chunks"
COMMUNITY_COLLECTION = "okf_communities"
DENSE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
SPARSE_MODEL = "qdrant/bm25"
LATE_INTERACTION_MODEL = "answerdotai/answerai-colbert-small-v1"
PREFETCH_LIMIT = 30

# switches
USE_VECTORS = True
USE_GRAPH = True
USE_COMMUNITIES = True
SPLIT_QUESTIONS = True
INCLUDE_NEW_FACTS = True

# knobs
N_VECTOR = 10        # chunks from Qdrant per query
N_FACTS = 15         # facts in the context
N_SEARCH = 10        # facts from plain fact search
N_CHUNKS = 8         # passages in the context
N_COMMUNITIES = 2    # community reports in the context
PER_PAPER = 3        # max passages per paper, unless the question names a paper
PAPER_BOOST = 1.5
RRF_K = 60           # the standard RRF constant (step 9)

for p in (FACTS_PATH, ENTITIES_PATH, CHUNKS_PATH):
    assert p.exists(), f"missing {p}"
if not REPORTS_PATH.exists():
    print("no community reports yet - run 05_graph_communities.ipynb; continuing without them")
    USE_COMMUNITIES = False
print("project:", PROJECT_ROOT)

project: c:\Users\Swapn\OneDrive\Documents\GitHub\OKF


In [2]:
# connect to Qdrant Cloud (credentials from .env, like 01_Chunking.ipynb)
load_dotenv(PROJECT_ROOT / ".env")
QDRANT_OK = REPORTS_IN_QDRANT = False
try:
    client = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"), cloud_inference=True, timeout=60)
    print(f"{CHUNK_COLLECTION}: {client.count(CHUNK_COLLECTION, exact=True).count} points")
    QDRANT_OK = True
    if client.collection_exists(COMMUNITY_COLLECTION):
        print(f"{COMMUNITY_COLLECTION}: {client.count(COMMUNITY_COLLECTION, exact=True).count} points")
        REPORTS_IN_QDRANT = True
    else:
        print(f"{COMMUNITY_COLLECTION} not found - run the upload in 05_graph_communities.ipynb; reports use keyword search for now")
except Exception as exc:
    print(f"!! Qdrant not reachable ({exc}).\n!! Vector search is OFF - this run uses only the graph (and keyword search for reports).")
USE_VECTORS = USE_VECTORS and QDRANT_OK

# check: Ollama
tags = requests.get(f"{OLLAMA_URL}/api/tags", timeout=10).json()
assert MODEL_NAME in [m["name"] for m in tags.get("models", [])], f"ollama pull {MODEL_NAME}"

okf_chunks: 2927 points
okf_communities: 130 points


## 2. Load the graph, chunks and community reports

In [3]:
facts = [json.loads(line) for line in FACTS_PATH.open(encoding="utf-8")]
entities = pd.read_csv(ENTITIES_PATH)
entities["aliases"] = entities["aliases"].map(ast.literal_eval)
with CHUNKS_PATH.open(encoding="utf-8") as f:
    chunk_by_id = {c["chunk_id"]: c for c in (json.loads(line) for line in f if line.strip())}
PAPER_TITLE = {c["document"]: c["title"] for c in chunk_by_id.values()}

reports = {}
if REPORTS_PATH.exists():
    with REPORTS_PATH.open(encoding="utf-8") as f:
        for row in (json.loads(line) for line in f if line.strip()):
            reports[row["id"]] = row                      # latest row per id wins
    reports = {i: r for i, r in reports.items() if not r["error"]}

# which reports each entity belongs to (leaf and facet reports; parents are reached through search)
reports_of = {}
for rid, r in reports.items():
    if r["kind"] != "parent":
        for m in r["members"]:
            reports_of.setdefault(m, []).append(rid)

G = nx.MultiDiGraph()
for e in entities.itertuples():
    G.add_node(e.name, type=e.type, aliases=e.aliases)
for i, f in enumerate(facts):
    G.add_edge(f["source"], f["target"], key=f["relation"], fact_id=i)

facts_of_chunk = {}                                       # chunk -> facts extracted from it
for i, f in enumerate(facts):
    for cid in f["chunk_ids"]:
        facts_of_chunk.setdefault(cid, []).append(i)

print(f"facts: {len(facts)} | entities: {G.number_of_nodes()} | chunks: {len(chunk_by_id)} | "
      f"community reports: {len(reports)} ({dict(Counter(r['kind'] for r in reports.values()))})")

facts: 2801 | entities: 2393 | chunks: 2927 | community reports: 130 ({'facet': 11, 'leaf': 108, 'parent': 11})


## 3. Entity linking and keyword search (same as `04_graph_query.ipynb`)

- `link(question)` finds entity names in the question; the longest match wins.
- `BM25` scores facts (`source relation target` + quotes), chunks, and community reports by keywords.
- `papers_named(question)` spots a paper named in the question ("the GPT-4 technical report", "HELM").

In [4]:
ALIASES = {
    "llm": "large language model", "llms": "large language model", "lm": "language model",
    "agi": "artificial general intelligence", "ai": "artificial intelligence",
    "helm": "Holistic Evaluation of Language Models",
    "rlhf": "reinforcement learning from human feedback", "cot": "chain-of-thought prompting",
    "icl": "in-context learning", "sft": "supervised fine-tuning", "peft": "parameter-efficient fine-tuning",
    "rag": "retrieval-augmented generation", "mllm": "multimodal large language model",
    "pii": "personally identifiable information", "nlp": "natural language processing",
}


def tidy(text):
    text = text.replace("“", '"').replace("”", '"').replace("’", "'").replace("‘", "'")
    text = text.replace("--", "-").replace("–", "-").replace("—", "-")
    return re.sub(r"\s+", " ", text).strip().lower()


surface_forms = {}
for name, data in G.nodes(data=True):
    for form in [name] + data["aliases"]:
        if len(tidy(form)) >= 2:
            surface_forms.setdefault(tidy(form), name)
for short, name in ALIASES.items():
    if name in G:
        surface_forms.setdefault(short, name)
PATTERNS = [(re.compile(r"(?<![a-z0-9])" + re.escape(form) + r"(?:s|es)?(?![a-z0-9])"), name)
            for form, name in surface_forms.items()]


def link(question):
    q = tidy(question)
    spans = sorted(((m.start(), m.end(), name) for rx, name in PATTERNS for m in rx.finditer(q)),
                   key=lambda s: (s[0] - s[1], s[0]))
    taken, seeds = [], []
    for start, end, name in spans:
        if any(start < t_end and t_start < end for t_start, t_end in taken):
            continue
        taken.append((start, end))
        if name not in seeds:
            seeds.append(name)
    return seeds


PAPER_HINTS = {
    "AGI": ["sparks of agi", "sparks of artificial general intelligence", "sparks paper"],
    "GPT-4": ["gpt-4 technical report", "gpt-4 report", "technical report", "system card"],
    "Holistic_Eval": ["helm", "holistic evaluation of language models"],
    "Foundation_Models": ["foundation models report", "opportunities and risks of foundation models"],
    "LLMs": ["llm survey", "survey of large language models"],
}


def papers_named(question):
    q = tidy(question)
    return [doc for doc, hints in PAPER_HINTS.items()
            if any(re.search(r"(?<![a-z0-9])" + re.escape(h) + r"(?![a-z0-9])", q) for h in hints)]


STOPWORDS = set('''a an the of and or to in on for with by from at as is are was were be been being it its this that these
those what which who whom how why when where do does did can could should would will may might about into than then
there their they them we our you your i me my he she his her not no yes vs'''.split())


def tokenize(text):
    return [t for t in re.findall(r"[a-z0-9]+(?:[-.][a-z0-9]+)*", text.lower()) if t not in STOPWORDS]


class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.tf = {key: Counter(tokenize(text)) for key, text in docs.items()}
        self.length = {key: sum(tf.values()) for key, tf in self.tf.items()}
        self.avg_len = sum(self.length.values()) / max(1, len(self.length))
        df = Counter(word for tf in self.tf.values() for word in tf)
        n = len(self.tf)
        self.idf = {w: math.log(1 + (n - d + 0.5) / (d + 0.5)) for w, d in df.items()}

    def scores(self, query, keys=None):
        q = tokenize(query)
        result = {}
        for key in (self.tf if keys is None else keys):
            tf = self.tf[key]
            norm = 1 - self.b + self.b * self.length[key] / self.avg_len
            s = sum(self.idf[w] * tf[w] * (self.k1 + 1) / (tf[w] + self.k1 * norm) for w in q if w in tf)
            if s > 0:
                result[key] = s
        return result


def report_text(r):
    return f"{r['title']}\n{r['summary']}\n" + "\n".join(x["text"] for x in r["findings"]) + "\n" + ", ".join(r["members"][:15])


fact_index = BM25({i: f"{f['source']} {f['relation'].replace('_', ' ').lower()} {f['target']}. " + " ".join(f["evidence"])
                   for i, f in enumerate(facts)})
chunk_index = BM25({cid: c["text"] for cid, c in chunk_by_id.items()})
report_index = BM25({rid: report_text(r) for rid, r in reports.items()}) if reports else None

q = "What are the three steps of RLHF, and how did RLHF affect GPT-4's calibration?"
print(link(q), papers_named("How does HELM measure calibration?"))

['calibration', 'GPT-4', 'reinforcement learning from human feedback'] ['Holistic_Eval']


## 4. Vector search in Qdrant

This is the exact query from `01_Chunking.ipynb`:
1. **prefetch**: the 30 best chunks by **dense** vectors (meaning), and the 30 best by **sparse** BM25 vectors (keywords)
2. **rerank** those ~60 candidates with **ColBERT** (`multi`), which compares every query token with every chunk token

Qdrant Cloud does the embedding (`models.Document(text=..., model=...)`), so nothing runs locally. The same query works on the community reports.

In [5]:
def qdrant_search(collection, query, top_k, id_field):
    result = client.query_points(
        collection,
        prefetch=[
            models.Prefetch(query=models.Document(text=query, model=DENSE_MODEL), using="dense", limit=PREFETCH_LIMIT),
            models.Prefetch(query=models.Document(text=query, model=SPARSE_MODEL), using="sparse", limit=PREFETCH_LIMIT),
        ],
        query=models.Document(text=query, model=LATE_INTERACTION_MODEL),
        using="multi",
        limit=top_k,
        with_payload=[id_field],
    )
    return [p.payload[id_field] for p in result.points]


def vector_chunks(query, top_k=None):
    return qdrant_search(CHUNK_COLLECTION, query, top_k or N_VECTOR, "chunk_id")


def vector_reports(query, top_k=5):
    return qdrant_search(COMMUNITY_COLLECTION, query, top_k, "community_id")


if QDRANT_OK:
    start = time.perf_counter()
    hits = vector_chunks("What did the GPT-4 technical report say about predictable scaling?")
    print(f"{(time.perf_counter() - start) * 1000:.0f} ms")
    for cid in hits[:5]:
        print("  ", cid, " > ".join(chunk_by_id[cid]["section_path"]))

617 ms
   GPT-4_0010 3 Predictable Scaling
   GPT-4_0009 3 Predictable Scaling
   GPT-4_0048 7 Conclusion
   GPT-4_0015 3 Predictable Scaling > 3.2 Scaling of Capabilities on HumanEval
   GPT-4_0016 3 Predictable Scaling > 3.2 Scaling of Capabilities on HumanEval


## 5. Two-part questions

"What are the three steps of RLHF, **and how did** RLHF affect GPT-4's calibration?" asks two things. One search for the whole sentence tends to find chunks for only one half. So we split at ", and how / what / why…", but **only if each half names an entity**. "Which papers discuss hallucination, and what does each one say about *it*?" is not split, because "it" can't be searched on its own.

Each half gets its own Qdrant search, and the result lists are fused in step 9.

In [6]:
SPLIT_RE = re.compile(r",?\s+and\s+(?=(?:how|what|why|which|who|when|where|does|did|is|are|was|were)\b)", re.IGNORECASE)


def split_question(question):
    parts = [p.strip(" ?,") for p in SPLIT_RE.split(question)]
    if len(parts) > 1 and all(link(p) for p in parts):
        return parts
    return []


for q in ["What are the three steps of RLHF, and how did RLHF affect GPT-4's calibration?",
          "Which papers discuss hallucination, and what does each one say about it?",
          "How does HELM measure calibration of language models?"]:
    print(f"{q}\n   -> {split_question(q) or '(not split)'}")

What are the three steps of RLHF, and how did RLHF affect GPT-4's calibration?
   -> ['What are the three steps of RLHF', "how did RLHF affect GPT-4's calibration"]
Which papers discuss hallucination, and what does each one say about it?
   -> (not split)
How does HELM measure calibration of language models?
   -> (not split)


## 6. Reciprocal Rank Fusion (RRF)

We get **ranked lists** from different searches: Qdrant for the question, Qdrant for each half, and the graph. Their scores aren't comparable (a ColBERT score of 19 and a BM25 score of 19 mean different things), so RRF ignores scores and uses **ranks**:

$$\text{RRF}(d) = \sum_{\text{lists}} \frac{1}{k + \text{rank}_\text{list}(d)} \qquad k = 60$$

An item that's #1 in one list gets 1/61. An item that's #3 in two lists gets 2/63, which is more. So **agreement between searches wins**. That's why RRF is the default fusion in hybrid search, including inside Qdrant itself.

In [7]:
def rrf(*ranked_lists, k=RRF_K):
    score = Counter()
    for ranked in ranked_lists:
        for rank, item in enumerate(ranked, start=1):
            score[item] += 1 / (k + rank)
    return score


demo = rrf(["a", "b", "c"], ["c", "d", "a"])
print({x: round(s, 4) for x, s in demo.most_common()})     # a and c are in both lists, so they win

{'a': 0.0323, 'c': 0.0323, 'b': 0.0161, 'd': 0.0161}


## 7. Graph retrieval, now also seeded by the vector hits

Same as `04_graph_query.ipynb`: walk from the question's entities and rank facts by BM25 × channel weight. There's one new channel:

| channel | meaning | weight |
|---|---|---|
| `bridge` | both ends are entities from the question | 2.0 |
| `hop 1` | one end is an entity from the question | 1.0 |
| `vector` | the fact was extracted from a chunk Qdrant found (**vector → graph**) | 0.8 |
| `search` | best BM25 match over all facts and their quotes | 0.7 |
| `hop 2` | touches a neighbour (only with `hops=2`) | 0.5 |

The `vector` channel is the classic hybrid pattern (Neo4j's "VectorCypher" retriever works the same way): **vectors find where to enter the graph, the graph adds what's connected.**

In [8]:
WEIGHT = {"bridge": 2.0, "hop 1": 1.0, "vector": 0.8, "search": 0.7, "hop 2": 0.5}


def graph_retrieve(queries, seeds, papers, vector_hits, hops=1, include_new=None):
    include_new = INCLUDE_NEW_FACTS if include_new is None else include_new
    allowed = lambda i: include_new or facts[i]["origin"] == "v1"
    channel = {}
    frontier, visited = set(seeds), set(seeds)
    for hop in range(1, hops + 1):
        nxt = set()
        for node in sorted(frontier):                            # sorted: same order every run
            for u, v, rel, d in list(G.out_edges(node, keys=True, data=True)) + list(G.in_edges(node, keys=True, data=True)):
                if allowed(d["fact_id"]) and d["fact_id"] not in channel:
                    channel[d["fact_id"]] = "bridge" if (u in seeds and v in seeds) else f"hop {hop}"
                nxt |= {u, v} - visited
        visited |= nxt
        frontier = nxt
    for cid in vector_hits[:5]:                                   # vector -> graph
        for i in facts_of_chunk.get(cid, []):
            if allowed(i) and i not in channel:
                channel[i] = "vector"

    bm25 = Counter()                                              # best score over the question and its parts
    for qx in queries:
        for i, s in fact_index.scores(qx).items():
            bm25[i] = max(bm25[i], s)
    added = 0
    for i, _ in bm25.most_common():
        if added == N_SEARCH:
            break
        if allowed(i) and i not in channel:
            channel[i] = "search"
            added += 1

    def score(i):
        s = bm25.get(i, 0) * WEIGHT[channel[i]]
        return s * PAPER_BOOST if papers and set(facts[i]["documents"]) & set(papers) else s

    top = sorted((i for i in channel if bm25.get(i, 0) > 0), key=score, reverse=True)[:N_FACTS]

    # the chunks behind the top facts, best keyword match first
    pool = list(dict.fromkeys(c for i in top for c in facts[i]["chunk_ids"] if c in chunk_by_id))
    chunk_bm25 = Counter()
    for qx in queries:
        for c, s in chunk_index.scores(qx, pool).items():
            chunk_bm25[c] = max(chunk_bm25[c], s)
    graph_chunks = sorted(pool, key=lambda c: chunk_bm25.get(c, 0), reverse=True)
    return [(i, channel[i], round(score(i), 1)) for i in top], graph_chunks

## 8. Community reports for the question

Two kinds of ranked lists, fused with RRF:
1. **search** over the reports: Qdrant (meaning + keywords) for the question and each part. If the reports aren't in Qdrant yet, local BM25 is used instead.
2. **membership**: the reports that contain the question's entities, ordered by BM25

The top `N_COMMUNITIES` go into the context as an overview.

In [9]:
def pick_reports(queries, seeds):
    if not reports:
        return []
    if REPORTS_IN_QDRANT:
        searched = [vector_reports(qx) for qx in queries]
    else:                                                          # keyword fallback
        searched = [[rid for rid, _ in Counter(report_index.scores(qx)).most_common(5)] for qx in queries]
    member_of = list(dict.fromkeys(rid for s in seeds for rid in reports_of.get(s, [])))
    keyword = report_index.scores(" ".join(queries), member_of)
    member_of.sort(key=lambda rid: keyword.get(rid, 0), reverse=True)
    fused = rrf(*searched, member_of)
    return [rid for rid, _ in fused.most_common(N_COMMUNITIES) if rid in reports]

## 9. Put the retrieval together

1. split the question (step 5)
2. Qdrant chunk search for the question and each part (step 4)
3. graph retrieval, entered from the question's entities *and* the vector hits (step 7)
4. **fuse the passages** with RRF: every Qdrant list + the graph's list. If the question names a paper, that paper's chunks get `PAPER_BOOST`×. Otherwise at most `PER_PAPER` chunks per paper.
5. community reports (step 8)

Each passage records **who found it** (`vector`, `graph`, or both), so you can see what each part contributed.

In [10]:
def retrieve(question, use_vectors=None, use_graph=None, use_communities=None, split=None, hops=1, include_new=None):
    use_vectors = USE_VECTORS if use_vectors is None else (use_vectors and QDRANT_OK)
    use_graph = USE_GRAPH if use_graph is None else use_graph
    use_communities = USE_COMMUNITIES if use_communities is None else use_communities
    split = SPLIT_QUESTIONS if split is None else split
    timing = {}

    seeds, papers = link(question), papers_named(question)
    parts = split_question(question) if split else []
    queries = [question] + parts

    t = time.perf_counter()
    vector_lists = [vector_chunks(qx) for qx in queries] if use_vectors else []
    timing["vector_ms"] = (time.perf_counter() - t) * 1000

    t = time.perf_counter()
    top_facts, graph_chunks = [], []
    if use_graph:
        vector_hits = list(dict.fromkeys(c for lst in vector_lists for c in lst))
        top_facts, graph_chunks = graph_retrieve(queries, seeds, papers, vector_hits, hops, include_new)
    timing["graph_ms"] = (time.perf_counter() - t) * 1000

    # fuse every vector list with the graph's list
    fused = rrf(*vector_lists, graph_chunks)
    if papers:
        for c in fused:
            if chunk_by_id[c]["document"] in papers:
                fused[c] *= PAPER_BOOST
    vector_set, graph_set = {c for lst in vector_lists for c in lst}, set(graph_chunks)

    def found_by(c):
        return "vector+graph" if c in vector_set and c in graph_set else ("vector" if c in vector_set else "graph")

    passages, per_paper = [], Counter()
    for c, s in fused.most_common():
        doc = chunk_by_id[c]["document"]
        if not papers and per_paper[doc] >= PER_PAPER:
            continue
        passages.append((c, found_by(c), round(s * 1000, 1)))      # RRF score x1000, easier to read
        per_paper[doc] += 1
        if len(passages) == N_CHUNKS:
            break

    t = time.perf_counter()
    communities = pick_reports(queries, seeds) if use_communities else []
    timing["community_ms"] = (time.perf_counter() - t) * 1000

    return {"question": question, "parts": parts, "seeds": seeds, "papers": papers, "facts": top_facts,
            "chunks": passages, "communities": communities, "timing": timing}


def show_retrieval(r, n_facts=6):
    print(f"seeds: {r['seeds'] or '(none)'}" + (f" | paper: {r['papers']}" if r["papers"] else "")
          + (f" | parts: {r['parts']}" if r["parts"] else ""))
    for rid in r["communities"]:
        print(f"   report  [{rid}] {reports[rid]['title']}")
    for i, how, s in r["facts"][:n_facts]:
        f = facts[i]
        print(f"   fact    {how:7} {s:5.1f}  {f['source']} --{f['relation']}--> {f['target']}")
    for c, who, s in r["chunks"]:
        print(f"   passage {who:13} {s:5.1f}  {c:22} {' > '.join(chunk_by_id[c]['section_path'])[:55]}")
    print("   timing: " + " | ".join(f"{k} {v:.0f}" for k, v in r["timing"].items()))


show_retrieval(retrieve("What are the three steps of RLHF, and how did RLHF affect GPT-4's calibration?"))

seeds: ['calibration', 'GPT-4', 'reinforcement learning from human feedback'] | parts: ['What are the three steps of RLHF', "how did RLHF affect GPT-4's calibration"]
   report  [C5.0] RLHF and Alignment in LLMs
   report  [C14] Tool Manipulation and Task Synthesis
   fact    bridge   40.6  GPT-4 --USES--> reinforcement learning from human feedback
   fact    bridge   22.3  reinforcement learning from human feedback --USES--> GPT-4
   fact    bridge   21.5  GPT-4 --MEASURES--> calibration
   fact    hop 1    12.4  GPT-4 --USES--> zero-shot prompting
   fact    hop 1    12.2  GPT-4 --OUTPERFORMS--> incorrect behavior
   fact    hop 1    12.1  GPT-4 --EVALUATED_ON--> exam benchmark
   passage vector+graph   49.2  GPT-4_0036             5 Limitations
   passage vector         47.4  AGI_0384               10.2 On the path to more general artificial intelligenc
   passage vector+graph   47.2  GPT-4_0086             B Impact of RLHF on capability
   passage vector+graph   45.3  GPT-4_0076   

## 10. Build the context and ask the LLM

The context has three parts, in order:
1. **Community reports**: the overview, cited as `[C3.1]`
2. **Facts**: the graph's map of what's related
3. **Passages**: the paper text, cited as `[GPT-4_0036]`

Blocks are added in that order until the 18,000-character budget is used.

In [11]:
def chunk_header(cid):
    c = chunk_by_id[cid]
    pages = f"p. {c['page_start']}" if c["page_start"] == c["page_end"] else f"pp. {c['page_start']}-{c['page_end']}"
    return f"{PAPER_TITLE[c['document']]} > {' > '.join(c['section_path'])}, {pages}"


def render_report(rid, max_chars=1500):
    r = reports[rid]
    findings = "\n".join(f"- {x['text']} [{', '.join(x['chunk_ids'])}]" for x in r["findings"])
    return f"### [{rid}] {r['title']}\n{r['summary']}\n{findings}"[:max_chars]


def render_fact(i):
    f = facts[i]
    quote = f["evidence"][0][:200] if f["evidence"] else ""
    return f"- {f['source']} --{f['relation']}--> {f['target']} [{', '.join(f['chunk_ids'][:3])}]\n  quote: \"{quote}\""


def build_context(r, budget=CONTEXT_CHARS):
    # returns the context text, the ids the model may cite, and the context as a list of blocks (for DeepEval)
    sections, size, cited, items = [], 0, set(), []

    def add(title, blocks):                     # blocks: (ids the block lets the model cite, text)
        nonlocal size
        kept = []
        for ids, block in blocks:
            if size + len(block) <= budget:
                kept.append(block)
                size += len(block)
                cited.update(ids)
        if kept:
            sections.append(f"## {title}\n\n" + "\n\n".join(kept))
            items.extend(["\n".join(kept)] if title.startswith("Facts") else kept)     # all facts = one block

    add("Community reports (overview, written by an LLM from the graph)",
        [([rid] + [c for x in reports[rid]["findings"] for c in x["chunk_ids"]], render_report(rid)) for rid in r["communities"]])
    add("Facts from the knowledge graph",
        [(facts[i]["chunk_ids"][:3], render_fact(i)) for i, _, _ in r["facts"]])
    add("Source passages",
        [([c], f"### [{c}] ({chunk_header(c)})\n{chunk_by_id[c]['text'].strip()}") for c, _, _ in r["chunks"]])
    return "\n\n".join(sections), cited, items


SYSTEM_PROMPT = '''You answer questions about five AI research papers using ONLY the knowledge below.
The knowledge has up to three parts:
- community reports: overviews written by an LLM from a knowledge graph, each starting with ### [report-id]
- facts: source --RELATION--> target, with a quote and the chunk ids it came from
- source passages: the paper text, each starting with ### [chunk-id]
Rules:
- Use only what the knowledge states. Do not add outside knowledge.
- Prefer the source passages for specific claims and numbers; reports and facts can be vague or wrong.
- After each claim, cite the id(s) you used in square brackets, e.g. [GPT-4_0036] or [C3.1].
- Keep numbers, model names and percentages exactly as written.
- If the knowledge does not contain the answer, say: "The knowledge does not contain this." and stop.
- Be concise: a short paragraph or a few bullets.'''


def ask_llm(question, context):
    start = time.perf_counter()
    response = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={
            "model": MODEL_NAME,
            "stream": False,
            "options": {"temperature": 0, "num_ctx": NUM_CTX, "num_predict": MAX_ANSWER_TOKENS},
            "messages": [{"role": "system", "content": SYSTEM_PROMPT},
                         {"role": "user", "content": f"Knowledge:\n\n{context}\n\n---\n\nQuestion: {question}"}],
        },
        timeout=600,
    )
    response.raise_for_status()
    data = response.json()
    answer = re.sub(r"<think>.*?</think>", "", data["message"]["content"], flags=re.S).strip()
    return answer, {"prompt_tokens": data.get("prompt_eval_count"), "answer_tokens": data.get("eval_count"),
                    "generation_ms": round((time.perf_counter() - start) * 1000, 1)}


start = time.perf_counter()
_ = ask_llm("Say OK.", "### [warmup]\nOK")
print(f"warm-up took {time.perf_counter() - start:.1f}s")

warm-up took 10.8s


## 11. `ask()`

Prints the retrieval (reports, facts, passages and who found each), the answer, the sources it cited, and timing (`retrieval_ms`, `generation_ms`). It returns the same keys as the other notebooks, plus `retrieval_context`: the exact blocks the model saw, which DeepEval grades from.

In [12]:
def sources_for(answer, cited):
    lines = []
    for x in sorted(cited, key=str):
        if f"[{x}" not in answer and x not in answer:
            continue
        if x in reports:
            lines.append(f"{x}  ->  community report: {reports[x]['title']} ({len(reports[x]['members'])} entities)")
        elif x in chunk_by_id:
            lines.append(f"{x}  ->  {chunk_header(x)}")
    return lines


def ask(question, show_context=False, **retrieve_kwargs):
    t0 = time.perf_counter()
    r = retrieve(question, **retrieve_kwargs)
    context, cited, items = build_context(r)
    retrieval_ms = (time.perf_counter() - t0) * 1000
    answer, stats = ask_llm(question, context)

    print(f"Q: {question}\n")
    show_retrieval(r)
    if show_context:
        print("\n--- context ---\n" + context + "\n--- end context ---")
    print("\nanswer:\n" + answer)
    print("\nsources cited:")
    for line in sources_for(answer, cited) or ["(none - check the answer!)"]:
        print("   " + line)
    print(f"\ntiming: retrieval {retrieval_ms:.0f} ms | generation {stats['generation_ms']:.0f} ms | "
          f"tokens in/out {stats['prompt_tokens']}/{stats['answer_tokens']} | context {len(context)} chars")
    return {"question": question, "answer": answer, "retrieved": r, "used": [c for c, _, _ in r["chunks"]],
            "retrieval_context": items, "retrieval_ms": round(retrieval_ms, 1), **stats}

## 12. What does each part add? (retrieval only, no LLM)

For each test question we know which **section** holds the answer. Below, every mode retrieves its passages, and we count how many come from that section:
- **vector only**: your Qdrant search alone, i.e. the `01_Chunking.ipynb` baseline
- **graph only**: what `04_graph_query.ipynb` does
- **hybrid**: both, fused, plus question splitting

This is a quick sanity check (7 questions, gold = "the right section"), not a benchmark.

In [13]:
def in_section(doc, pattern):
    return {c for c, x in chunk_by_id.items() if x["document"] == doc and re.search(pattern, " > ".join(x["section_path"]))}


GOLD = {
    "How does HELM measure calibration of language models?": in_section("Holistic_Eval", r"4\.4 Calibration|C\.2 Calibration"),
    "What did the GPT-4 technical report say about predictable scaling?": in_section("GPT-4", r"^3 Predictable"),
    "What is homogenization in foundation models and why is it risky?": in_section("Foundation_Models", r"1\.1 Emergence and homogenization|5\.6\.1 Homogenization"),
    "How did the Sparks of AGI paper test GPT-4 on theory of mind?": in_section("AGI", r"^6\.1"),
    "What are the three steps of RLHF, and how did RLHF affect GPT-4's calibration?": {"GPT-4_0036"} | in_section("LLMs", r"5\.2\.3"),
    "Which model was most accurate on TruthfulQA in HELM?": {"Holistic_Eval_0038"},
    "What score did GPT-4 get on the bar exam?": {c for c, x in chunk_by_id.items() if x["document"] == "GPT-4"
                                                   and re.search(r"bar exam", x["text"], re.I) and re.search(r"top 10%|298|90th", x["text"])},
}
MODES = {
    "vector only": dict(use_vectors=True, use_graph=False, use_communities=False, split=False),
    "graph only": dict(use_vectors=False, use_graph=True, use_communities=False, split=False),
    "hybrid": dict(),
}

rows = []
for question, gold in GOLD.items():
    row = {"question": question[:55]}
    for mode, kwargs in MODES.items():
        got = [c for c, _, _ in retrieve(question, **kwargs)["chunks"]]
        row[mode] = len(set(got) & gold)
    rows.append(row)
check = pd.DataFrame(rows)
display(check)
print("questions with at least one passage from the right section:",
      {mode: f"{(check[mode] > 0).sum()}/{len(check)}" for mode in MODES})

,question,vector only,graph only,hybrid
0,How does HELM measure calibration of language ...,0,2,1
1,What did the GPT-4 technical report say about ...,7,0,6
2,What is homogenization in foundation models an...,3,2,3
3,How did the Sparks of AGI paper test GPT-4 on ...,8,1,6
4,"What are the three steps of RLHF, and how did ...",1,1,2
5,Which model was most accurate on TruthfulQA in...,0,1,1
6,What score did GPT-4 get on the bar exam?,2,2,2


questions with at least one passage from the right section: {'vector only': '5/7', 'graph only': '6/7', 'hybrid': '7/7'}


## 13. Ask the seven questions

Same questions and order as `02_OKF_query.ipynb` and `04_graph_query.ipynb`.

In [14]:
r1 = ask("How does HELM measure calibration of language models?")

Q: How does HELM measure calibration of language models?

seeds: ['language model', 'calibration', 'Holistic Evaluation of Language Models'] | paper: ['Holistic_Eval']
   report  [C7.4] Calibration in Model Evaluation
   report  [C7.1] Holistic Evaluation of Language Models
   fact    hop 1    21.1  calibration --MEASURES--> classification task
   fact    hop 1    21.1  calibration --MEASURES--> selective classification
   fact    hop 1    17.1  Holistic Evaluation of Language Models --MEASURES--> transparency
   fact    hop 1    15.6  Holistic Evaluation of Language Models --COVERS--> Reddit
   fact    hop 1    15.6  Holistic Evaluation of Language Models --COVERS--> Swahili
   fact    hop 1    15.6  Holistic Evaluation of Language Models --COVERS--> elderly
   passage vector+graph   45.9  Holistic_Eval_0009     1 Introduction
   passage vector         24.6  Holistic_Eval_0014     1 Introduction > 1.1 HELM
   passage graph          24.6  Holistic_Eval_0238     8 Experiments and result

In [15]:
r2 = ask("What did the GPT-4 technical report say about predictable scaling?")

Q: What did the GPT-4 technical report say about predictable scaling?

seeds: ['predictable scaling', 'technical report', 'GPT-4'] | paper: ['GPT-4']
   report  [C6.2] GPT-4 Models and OpenAI Research
   report  [C0.0:risks] GPT-4 Risks and Limitations
   fact    bridge   37.7  GPT-4 --USES--> predictable scaling
   fact    hop 1    25.8  GPT-4 --USES--> scaling
   fact    hop 1    18.5  GPT-4 --EVALUATED_ON--> OpenAI technical report on GPT-4
   fact    hop 1    17.5  predictable scaling --BUILDS_ON--> deep learning stack
   fact    hop 1    16.8  technical report --DEVELOPED_BY--> OpenAI
   fact    hop 1    16.1  GPT-4 --IS_A--> large multimodal model
   passage vector+graph   48.0  GPT-4_0048             7 Conclusion
   passage vector+graph   31.1  LLMs_0068              2 OVERVIEW > 2.2 Technical Evolution of GPT-series Mode
   passage vector         24.6  GPT-4_0010             3 Predictable Scaling
   passage vector         24.2  GPT-4_0009             3 Predictable Scaling
   pa

In [16]:
r3 = ask("What is homogenization in foundation models and why is it risky?")

Q: What is homogenization in foundation models and why is it risky?

seeds: ['foundation model', 'homogenization']
   report  [C2.6] Emergence and Homogenization in AI Research
   report  [C2.0:risks] Foundation Models and Equity Risks
   fact    hop 1    11.1  homogenization --IS_A--> emergence
   fact    hop 1     9.5  BERT --HAS_LIMITATION--> homogenization
   fact    hop 1     9.3  homogenization --PART_OF--> AI research
   fact    hop 1     7.6  multilingual foundation models --BUILDS_ON--> foundation model
   fact    hop 1     7.3  foundation model --IS_A--> general-purpose technology
   fact    hop 1     7.2  foundation model --IS_A--> task-agnostic
   passage vector+graph   32.8  Foundation_Models_0019 1 INTRODUCTION > 1.1 Emergence and homogenization
   passage vector+graph   32.0  Foundation_Models_0006 1 INTRODUCTION > 1.1 Emergence and homogenization
   passage vector         16.1  Foundation_Models_0020 1 INTRODUCTION > 1.1 Emergence and homogenization
   passage graph    

In [17]:
r4 = ask("How did the Sparks of AGI paper test GPT-4 on theory of mind?")

Q: How did the Sparks of AGI paper test GPT-4 on theory of mind?

seeds: ['GPT-4', 'artificial general intelligence'] | paper: ['AGI']
   report  [C0.0:capabilities] GPT-4 Capabilities Across Domains
   report  [C0.0:risks] GPT-4 Risks and Limitations
   fact    bridge   29.3  GPT-4 --IS_A--> artificial general intelligence
   fact    search   22.6  Sally-Anne test --IS_A--> false-belief test
   fact    bridge   21.6  GPT-4 --HAS_RISK--> artificial general intelligence
   fact    hop 1    20.2  GPT-4 --HAS_CAPABILITY--> graph theory
   fact    hop 1    16.7  GPT-4 --HAS_CAPABILITY--> algorithm
   fact    hop 1    12.0  GPT-4 --HAS_LIMITATION--> deep understanding
   passage vector+graph   44.7  AGI_0222               6.1 Understanding Humans: Theory of Mind > 6.1.2 Testin
   passage vector         24.6  AGI_0228               6.1 Understanding Humans: Theory of Mind > 6.1.2 Testin
   passage graph          24.6  AGI_0217               6.1 Understanding Humans: Theory of Mind > 6.1.1 Te

In [18]:
r5 = ask("Which papers discuss hallucination, and what does each one say about it?")

Q: Which papers discuss hallucination, and what does each one say about it?

seeds: ['hallucination']
   report  [C0.3] Hallucination Detection in LLMs
   report  [C0] GPT-4 Capabilities and Reasoning
   fact    hop 1     6.9  extrinsic hallucination --IS_A--> hallucination
   fact    hop 1     6.9  intrinsic hallucination --IS_A--> hallucination
   fact    hop 1     6.4  factual hallucination --IS_A--> hallucination
   fact    hop 1     6.1  large language model --HAS_LIMITATION--> hallucination
   fact    hop 1     6.1  large language model --HAS_RISK--> hallucination
   fact    hop 1     5.8  uncertainty estimation --USES--> hallucination
   passage vector+graph   32.5  LLMs_0682              9 ADVANCED TOPICS > 9.7 Hallucination
   passage vector+graph   31.5  LLMs_0683              9 ADVANCED TOPICS > 9.7 Hallucination > 9.7.1 Definitio
   passage vector+graph   30.7  LLMs_0438              7 CAPACITY AND EVALUATION > 7.1 Basic Ability > 7.1.2 K
   passage graph          14.7  AGI

In [19]:
r6 = ask("What are the three steps of RLHF, and how did RLHF affect GPT-4's calibration?")

Q: What are the three steps of RLHF, and how did RLHF affect GPT-4's calibration?

seeds: ['calibration', 'GPT-4', 'reinforcement learning from human feedback'] | parts: ['What are the three steps of RLHF', "how did RLHF affect GPT-4's calibration"]
   report  [C5.0] RLHF and Alignment in LLMs
   report  [C14] Tool Manipulation and Task Synthesis
   fact    bridge   40.6  GPT-4 --USES--> reinforcement learning from human feedback
   fact    bridge   22.3  reinforcement learning from human feedback --USES--> GPT-4
   fact    bridge   21.5  GPT-4 --MEASURES--> calibration
   fact    hop 1    12.4  GPT-4 --USES--> zero-shot prompting
   fact    hop 1    12.2  GPT-4 --OUTPERFORMS--> incorrect behavior
   fact    hop 1    12.1  GPT-4 --EVALUATED_ON--> exam benchmark
   passage vector+graph   49.2  GPT-4_0036             5 Limitations
   passage vector         47.4  AGI_0384               10.2 On the path to more general artificial intelligenc
   passage vector+graph   47.2  GPT-4_0086      

In [20]:
r7 = ask("How many parameters does Gemini Ultra have?")

Q: How many parameters does Gemini Ultra have?

seeds: (none)
   report  [C4] LLaMA Model Lineage and Instruction Datasets
   report  [C6.4] T0++ and Cohere Model Lineage
   fact    search    7.5  T0++ --OUTPERFORMS--> Anthropic-LM v4-s3
   fact    search    6.0  GPT-3 --HAS_CAPABILITY--> natural language
   fact    search    5.1  perplexity --MEASURES--> downstream use case
   fact    search    5.1  fine-tuning --USES--> memory
   fact    search    4.9  GPT-4 --HAS_CAPABILITY--> proof that there are infinitely many primes
   fact    search    4.9  foundation model --IS_A--> many models
   passage vector+graph   30.6  Foundation_Models_0004 1 INTRODUCTION
   passage vector+graph   30.3  Holistic_Eval_0224     7 Adaptation via prompting
   passage vector         16.4  AGI_0297               8.1 Warm-up with two basic examples
   passage graph          16.4  Foundation_Models_0360 4 TECHNOLOGY > 4.3 Adaptation > 4.3.1 Methods for found
   passage vector         16.1  AGI_0088            

In [21]:
summary = pd.DataFrame([{k: r[k] for k in ("question", "retrieval_ms", "generation_ms", "prompt_tokens", "answer_tokens")}
                        for r in (r1, r2, r3, r4, r5, r6, r7)])
summary["question"] = summary["question"].str[:60]
display(summary)

,question,retrieval_ms,generation_ms,prompt_tokens,answer_tokens
0,How does HELM measure calibration of language ...,883.4,8288.4,3633,228
1,What did the GPT-4 technical report say about ...,1572.1,8629.7,3221,242
2,What is homogenization in foundation models an...,1700.8,7355.6,2254,202
3,How did the Sparks of AGI paper test GPT-4 on ...,1701.3,8426.8,3772,218
4,"Which papers discuss hallucination, and what d...",1554.5,15138.4,3191,576
5,"What are the three steps of RLHF, and how did ...",3146.8,9565.6,3740,297
6,How many parameters does Gemini Ultra have?,1789.2,4333.4,3945,8


## 14. Your turn

In [22]:
result = ask("Which model was most accurate on TruthfulQA in HELM?")

Q: Which model was most accurate on TruthfulQA in HELM?

seeds: ['TruthfulQA', 'Holistic Evaluation of Language Models'] | paper: ['Holistic_Eval']
   report  [C8] Evolution and Evaluation of Language Models
   report  [C8.1] TruthfulQA and Evaluation Metrics
   fact    search   13.1  text-davinci-002 --HAS_CAPABILITY--> question answering
   fact    hop 1    11.1  TruthfulQA --PART_OF--> language modeling
   fact    search   10.9  text-davinci-002 --HAS_LIMITATION--> morphology
   fact    search   10.9  text-davinci-002 --HAS_LIMITATION--> semantic
   fact    search   10.5  T0++ --OUTPERFORMS--> TNLG v2
   fact    hop 1    10.5  multiple choice (joint) --USES--> TruthfulQA
   passage vector+graph   45.6  Holistic_Eval_0254     8 Experiments and results > 8.1 Meta-analysis
   passage vector         24.6  Holistic_Eval_0310     8 Experiments and results > 8.4 Targeted evaluations
   passage graph          24.6  Holistic_Eval_0309     8 Experiments and results > 8.4 Targeted evaluations


## Save results for the comparison

This runs every question in `data/eval/eval_questions.jsonl` through `ask()` (quietly) and saves one line per question to `artifacts/results/graph_hybrid.jsonl`. Each line holds:
- the **answer**
- **`retrieval_context`**: the exact blocks the model saw, in order
- **timing**: `retrieval_ms`, `generation_ms`
- **tokens**: `prompt_tokens`, `answer_tokens`

`08_evaluate_methods.ipynb` grades all four methods from these files with DeepEval, using one judge, so every method is judged the same way. Re-run this cell whenever you change the method or the questions.

In [23]:
import contextlib
import io

METHOD = "graph_hybrid"
EVAL_PATH = PROJECT_ROOT / "data" / "eval" / "eval_questions.jsonl"
RESULTS_PATH = PROJECT_ROOT / "artifacts" / "results" / f"{METHOD}.jsonl"
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

# a hybrid result only counts if its vector parts are really on (otherwise this would save a keyword-only run)
assert QDRANT_OK and USE_VECTORS, "Qdrant isn't connected or USE_VECTORS is off - fix that before saving results"
assert REPORTS_IN_QDRANT, "community reports aren't in Qdrant - run 05_graph_communities.ipynb first"
print(f"switches: USE_VECTORS={USE_VECTORS} USE_GRAPH={USE_GRAPH} USE_COMMUNITIES={USE_COMMUNITIES} "
      f"SPLIT_QUESTIONS={SPLIT_QUESTIONS} INCLUDE_NEW_FACTS={INCLUDE_NEW_FACTS}")

eval_questions = [json.loads(line) for line in EVAL_PATH.open(encoding="utf-8") if line.strip()]
print(f"{len(eval_questions)} questions -> {RESULTS_PATH}")


def ask_quietly(question, attempts=2):
    # one retry, in case Ollama or Qdrant hiccups
    for attempt in range(attempts):
        try:
            with contextlib.redirect_stdout(io.StringIO()):          # ask() prints a lot; keep this cell readable
                return ask(question)
        except Exception as exc:
            if attempt == attempts - 1:
                raise
            print(f"   failed ({type(exc).__name__}: {str(exc)[:80]}), retrying in 10 s")
            time.sleep(10)


# written line by line: if it stops halfway, the answers so far are kept (08 shows the rest as missing)
with RESULTS_PATH.open("w", encoding="utf-8") as f:
    for n, item in enumerate(eval_questions, start=1):
        r = ask_quietly(item["question"])
        row = {"id": item["id"], "method": METHOD, "question": item["question"], "answer": r["answer"],
               "retrieval_context": r["retrieval_context"], "retrieval_ms": r["retrieval_ms"],
               "generation_ms": r["generation_ms"], "prompt_tokens": r["prompt_tokens"],
               "answer_tokens": r["answer_tokens"]}
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
        f.flush()
        print(f"[{n:2}/{len(eval_questions)}] {item['id']:5} retrieval {r['retrieval_ms']:6.0f} ms | "
              f"generation {r['generation_ms']:6.0f} ms | {r['answer'][:60]!r}")
print(f"saved {len(eval_questions)} results to {RESULTS_PATH.name}")

switches: USE_VECTORS=True USE_GRAPH=True USE_COMMUNITIES=True SPLIT_QUESTIONS=True INCLUDE_NEW_FACTS=True
30 questions -> c:\Users\Swapn\OneDrive\Documents\GitHub\OKF\artifacts\results\graph_hybrid.jsonl
[ 1/30] q01   retrieval   1656 ms | generation   4248 ms | 'text-davinci-002 was most accurate on TruthfulQA in HELM, wi'
[ 2/30] q02   retrieval    968 ms | generation   5997 ms | 'The knowledge does not contain the exact score that GPT-4 go'
[ 3/30] q03   retrieval   3433 ms | generation   7254 ms | "In the Sparks of AGI paper, GPT-4's accuracy on arithmetic e"
[ 4/30] q04   retrieval   3514 ms | generation   6555 ms | "GPT-4's safety mitigations reduced its tendency to respond t"
[ 5/30] q05   retrieval   3966 ms | generation   5543 ms | "LLaMA's pre-training data mixture consists of: over 80% webp"
[ 6/30] q06   retrieval   1634 ms | generation   6847 ms | 'The knowledge does not contain the specific memory usage for'
[ 7/30] q07   retrieval   1537 ms | generation   7350 ms | 'HEL

## 15. Experiments worth trying

1. **One part at a time:** `ask(q, use_graph=False, use_communities=False)` is plain vector RAG. `ask(q, use_vectors=False, use_communities=False)` is `04_graph_query`. Compare the answers, not just the passages.
2. **Reports only:** `ask(q, use_vectors=False, use_graph=False)` answers from community reports alone. That's closest to GraphRAG's "global search". Try it on the hallucination question.
3. **No splitting:** `ask(q6, split=False)` on the RLHF question. Does the "three steps" half get lost?
4. **Only v1 facts:** `INCLUDE_NEW_FACTS = False` in cell 1, once you've graded `review_graph_v2.csv`.
5. **See what the model saw:** `ask(q, show_context=True)`.

### Things to know
- The vector parts need Qdrant Cloud. If the connection fails, the notebook says so and runs graph-only, so check the first cells' output before comparing results.
- Community reports are written by qwen3-4b from extracted facts, so they're two steps away from the paper text. Their findings cite chunk IDs, so you can check them.
- Fairness: this notebook has vectors and OKF_query doesn't. If hybrid wins, part of the win may be the embeddings, not the graph. Section 12's "vector only" column is there to tell those apart.